In [0]:
import pandas as pd
import time
from pyspark.sql import functions as f
from pyspark.sql import types as t
import random
from src.utils.validations import *
from src.utils.file import *
from datetime import datetime
from pyspark.sql import SparkSession






def quality_apply(dir_path):

    data_hoje = datetime.now().strftime("%Y%m%d")
    dir_path = r"/Volumes/workspace/callcenter/disc_volumes/landing/"
    path_processing = r""

    arquivos = dbutils.fs.ls(dir_path)

    for arquivo in arquivos:

        nm_arquivo = arquivo.name

        if not nm_arquivo.startswith(f"discagem_{data_hoje}_"):
            continue

        try:

            # =====================================
            # LEITURA
            # =====================================
            df = fr.read_csv(
                path=arquivo.path,
                header=True,
                use_spark=True,
                sep=";"
            )

            # =====================================
            # QUALITY
            # =====================================
            valido = Quality.validar_arquivo(
                df=df,
                nm_arquivo=nm_arquivo
            )

            # =====================================
            # APROVADO
            # =====================================
            if valido:

                fm.move_to_approved(
                    source_path=arquivo.path
                )

            # =====================================
            # REPROVADO
            # =====================================
            else:

                fm.move_to_rejected(
                    source_path=arquivo.path
                )

        except Exception as e:

            fm.move_to_rejected(
                source_path=arquivo.path
            )

            print(e)
